In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession


ss = SparkSession.builder \
    .master("local[*]") \
    .appName("Domain_Shift_Analysis") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("Spark Session started")

In [ ]:
input_folder_path = "/content/drive/MyDrive/DDAM project/data/risultato_pca_pulito_finale.csv"

df = ss.read.csv(
    input_folder_path,
    header=True,
    inferSchema=True,
    sep=","
)

print(f"Total rows: {df.count()}")

In [ ]:
from pyspark.sql.functions import col, lit

# Define the Central America satellite Tiles (Source Domain)
source_domain_tiles = [
    "16PCC", "16PDC", "16PEC", "16QED", # Honduras e Guatemala
    "19QDA",                            # Santo Domingo
    "18QWF", "18QYF", "18QYG"           # Haiti
]

# Source domain - Training set
df_train = df.filter(col("Tile").isin(source_domain_tiles))

# Target Domain - Test set (Rest of the World)
df_test = df.filter(~col("Tile").isin(source_domain_tiles))

print(f"Source domain size (Train): {df_train.count()} rows")
print(f"Target Domain size (Test): {df_test.count()} rows")

In [ ]:
columns_to_drop = ["Tile", "XCoords", "YCoords", "Image", "Date"]

cols_to_drop_train = [c for c in columns_to_drop if c in df_train.columns]

df_train = df_train.drop(*cols_to_drop_train)
df_test = df_test.drop(*cols_to_drop_train)

print("Geographic variables (Tiles and Coordinates) removed to prevent data leakage.")

In [ ]:
# OPTIONAL CELL: Transformation to Binary Classification
# If this cell is executed, the task becomes Binary (Marine Debris vs Other).
# If it is commented out or skipped, the task remains Multi-Class.

# from pyspark.sql.functions import col, when

# # Rename everything that is NOT "Marine Debris" to "Other"
# df_train = df_train.withColumn(
#     "Class",
#     when(col("Class") == "Marine Debris", "Marine Debris").otherwise("Other")
# )

# df_test = df_test.withColumn(
#     "Class",
#     when(col("Class") == "Marine Debris", "Marine Debris").otherwise("Other")
# )

# print("Binary transformation applied! The classes are now:", df_train.select("Class").distinct().rdd.flatMap(lambda x: x).collect())

In [ ]:
from itertools import chain
from pyspark.sql.functions import create_map

# Calculate the frequencies of each class in the training set
class_counts = df_train.groupBy("Class").count().collect()
total_train = df_train.count()

# Calculate the weight: (Total Rows) / (Number of classes * Specific class count)
# We use len(class_counts) to balance the global impact
num_classes = len(class_counts)
weight_map = {row['Class']: float(total_train) / (num_classes * row['count']) for row in class_counts}

print("Weights:")
for cls, w in sorted(weight_map.items(), key=lambda item: item[1], reverse=True):
    print(f" - {cls}: {w:.4f}")

mapping_expr = create_map([lit(x) for x in chain(*weight_map.items())])

df_train = df_train.withColumn("class_weight", mapping_expr[col("Class")])

df_test = df_test.withColumn("class_weight", lit(1.0))

df_train.cache()
df_test.cache()

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from xgboost.spark import SparkXGBClassifier
from pyspark.ml import Pipeline

confidence_indexer = StringIndexer(inputCol="Confidence", outputCol="Confidence_indexed", handleInvalid="keep")
confidence_ohe = OneHotEncoder(inputCol="Confidence_indexed", outputCol="Confidence_ohe", dropLast=False)

label_indexer = StringIndexer(inputCol="Class", outputCol="label")

feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

xgb_estimator = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    # weight_col="class_weight", # uncomment this line if you want to weight the classes, comment if you don't want it
    num_workers=2,
    seed=42
)

pipeline = Pipeline(stages=[confidence_indexer, confidence_ohe, label_indexer, assembler, xgb_estimator])

In [ ]:
# Setup Random Search
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import random

large_param_grid = (
    ParamGridBuilder()
    .addGrid(xgb_estimator.max_depth, [4, 6, 8, 16])
    .addGrid(xgb_estimator.learning_rate, [0.05, 0.1, 0.2])
    .addGrid(xgb_estimator.subsample, [0.7, 0.8, 1.0])
    .addGrid(xgb_estimator.n_estimators, [50, 100])
    .build()
)

random.seed(42)
sampled_paramGrid = random.sample(large_param_grid, 8) #number of combinations

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

# Setup CrossValidator
crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=sampled_paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

In [ ]:
cv_model = crossval.fit(df_train)
print("Training completed")

#Best model
best_pipeline = cv_model.bestModel
best_xgb = best_pipeline.stages[-1]

print("\nBest Hyperparameters:")
print(f" - Max Depth: {best_xgb.getOrDefault('max_depth')}")
print(f" - Learning Rate: {best_xgb.getOrDefault('learning_rate')}")
print(f" - Subsample: {best_xgb.getOrDefault('subsample')}")
print(f" - N Estimators: {best_xgb.getOrDefault('n_estimators')}")

In [ ]:
from pyspark.ml.feature import IndexToString

predictions = cv_model.transform(df_test)
predictions.cache()

#Global metrics
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
f1_macro = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})

print("TARGET DOMAIN GLOBAL METRICS")
print(f"Accuracy : {accuracy:.4f}")
print(f"F1-Score (Weighted): {f1_macro:.4f}\n")

label_converter = IndexToString(
    inputCol="prediction",
    outputCol="predictedLabel",
    labels=best_pipeline.stages[2].labels
)
predictions_with_strings = label_converter.transform(predictions)

print("Confusion matrix:")
confusion_matrix = predictions_with_strings.groupBy('Class').pivot('predictedLabel').count().fillna(0).orderBy('Class')
confusion_matrix.show()

In [ ]:
from pyspark.mllib.evaluation import MulticlassMetrics

prediction_and_labels = predictions.select(col("prediction"), col("label")).rdd.map(lambda row: tuple(map(float, row)))

metrics = MulticlassMetrics(prediction_and_labels)
labels = best_pipeline.stages[2].labels

print(f"{'Classs':<25} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 70)

for idx, label_str in enumerate(labels):
    label_idx = float(idx)

    try:
        class_precision = metrics.precision(label=label_idx)
        class_recall = metrics.recall(label=label_idx)
        class_f1 = metrics.fMeasure(label=label_idx)

        print(f"{label_str:<25} {class_precision:<15.4f} {class_recall:<15.4f} {class_f1:<15.4f}")
    except:
        print(f"{label_str:<25} {'N/A':<15} {'N/A':<15} {'N/A':<15}")

predictions.unpersist()

In [ ]:
# #CONFUSION MATRIX - only for the binary case - uncomment if needed

# import matplotlib.pyplot as plt
# import seaborn as sns
# import pandas as pd
# from pyspark.sql.functions import col

# cm_pd = confusion_matrix.toPandas()

# cm_pd.set_index('Class', inplace=True)

# cm_pd = cm_pd[['Marine Debris', 'Other']]
# cm_pd = cm_pd.reindex(['Marine Debris', 'Other'])

# FONT_SIZE = 22

# plt.figure(figsize=(7, 6))

# ax = sns.heatmap(
#     cm_pd,
#     annot=True,
#     fmt='d',
#     cmap='Blues',
#     cbar=False,
#     annot_kws={"size": FONT_SIZE + 4, "weight": "bold"}
# )

# ax.set_xlabel('Predicted Label', fontsize=FONT_SIZE, fontweight='bold', labelpad=15)
# ax.set_ylabel('True Label', fontsize=FONT_SIZE, fontweight='bold', labelpad=15)


# ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
# plt.xticks(rotation=0)
# plt.yticks(rotation=0)

# plt.tight_layout()
# plt.savefig('/content/drive/MyDrive/DDAM project/data/confusion_matrix_binary.pdf', format='pdf', bbox_inches='tight')
# plt.show()

In [ ]:
# # ROC CURVE - only for the binary case - uncomment if needed

# import matplotlib.pyplot as plt
# from pyspark.ml.functions import vector_to_array
# from sklearn.metrics import roc_curve, auc

# preds_for_roc_spark = predictions.select(
#     col("label"),
#     vector_to_array(col("probability")).getItem(1).alias("prob_positive")
# )

# preds_pdf = preds_for_roc_spark.toPandas()

# fpr, tpr, thresholds = roc_curve(preds_pdf['label'], preds_pdf['prob_positive'])
# auc_value = auc(fpr, tpr)

# FONT_SIZE = 22

# plt.figure(figsize=(7, 6))

# plt.plot(fpr, tpr, color='#2B3990', linewidth=4, label=f'AUC = {auc_value:.3f}')

# plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=2)

# plt.xlim([-0.02, 1.0])
# plt.ylim([0.0, 1.02])

# plt.xlabel('False Positive Rate', fontsize=FONT_SIZE, fontweight='bold')
# plt.ylabel('True Positive Rate', fontsize=FONT_SIZE, fontweight='bold')

# plt.xticks(fontsize=FONT_SIZE)
# plt.yticks(fontsize=FONT_SIZE)

# plt.legend(loc="lower right", fontsize=FONT_SIZE)

# plt.tight_layout()
# # plt.savefig('/content/drive/MyDrive/DDAM project/data/roc_curve_binary.pdf', format='pdf', bbox_inches='tight')
# plt.show()